# Inspect latest DC4 game log

Fetches the most recent match from the DigitalCurling4 server's REST API and draws the sheet after every shot.
Server URL: `DC_SERVER_URL`, from the environment or the repo's `.env`.

Based on the DC3 notebooks in `../digitalcurling-deeplearning` (`convert_log.ipynb`, `training_light_fixsize.ipynb`).
Differences from DC3:
- DC3 read `game.dcl2` JSON-lines files; DC4 keeps the log in the server DB, read via the read-only GET endpoints
  (see `dc4learning.api`).
- Coordinates are meters: `x` across the sheet (0 = center line), `y` along the sheet, tee at `y = 38.405`.
  A stone at `(0, 0)` is not in play (not thrown yet, or removed).
- Sheet orientation is the same as the DC3 plots: delivery runs down the page and +x is on the left (top-down view).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import rcParams

from dc4learning.api import DC4Client, team_maps
from dc4learning.sheet import EDGE, OTHER_TEAM, TEAM_COLOR, changed_stone, draw_sheet, draw_stones, to_sheet_grid

rcParams['font.family'] = 'sans-serif'
rcParams['text.usetex'] = False

client = DC4Client()
client.base_url

## Latest matches

In [ ]:
matches = client.list_matches()
matches.head(10)

In [ ]:
# 0 = latest match. Or set MATCH_ID to a specific uuid string.
MATCH_INDEX = 6
MATCH_ID = None

match_id = MATCH_ID or matches.iloc[MATCH_INDEX]['match_id']
match = client.get_match(match_id)
team_names, team_by_id = team_maps(match)
print(match_id, match['match_name'], match['started_at'], team_names)

## Load states

One row per sheet position. Plain shot columns = shot thrown *from* this state; `prev_*` = shot that produced it.

In [ ]:
states = client.load_states(match_id)
print(len(states), 'states,', states['end_number'].nunique(), 'ends')
states[['end_number', 'team_shot_number', 'total_shot_number', 'created_at',
        'translational_velocity', 'angular_velocity', 'shot_angle']]

## Score board

In [ ]:
final = states.iloc[-1]
score = match.get('score') or {'team0': final['score_team0'], 'team1': final['score_team1']}
n_ends = max(len(score['team0'] or []), len(score['team1'] or []))
board = pd.DataFrame(
    [list(score['team0'] or []), list(score['team1'] or [])],
    index=[f"team0: {team_names['team0']}", f"team1: {team_names['team1']}"],
    columns=[f'E{i+1}' for i in range(n_ends)],
)
board['Total'] = board.sum(axis=1)
winner = team_by_id.get(match['winner_team_id'] or final['winner_team_id'])
print('winner:', winner, team_names.get(winner))
board

## Single position

In [ ]:
def shot_label(row):
    if pd.isna(row['prev_translational_velocity']):
        return ''
    return (f"v={row['prev_translational_velocity']:.3f} θ={np.degrees(row['prev_shot_angle']):.2f}°"
            f" {'ccw' if row['prev_angular_velocity'] > 0 else 'cw'}")


def show_state(end, total_shot, ax=None):
    rows = states[(states.end_number == end) & (states.total_shot_number == total_shot)]
    if rows.empty:
        raise ValueError(f'no state for end {end} shot {total_shot}')
    row = rows.iloc[-1]
    if ax is None:
        _, ax = plt.subplots(figsize=(3.6, 6.4))
    draw_sheet(ax)
    draw_stones(ax, row['stones'])
    ax.set_title(f"End {end + 1}  shot {total_shot}\n{team_by_id.get(row['prev_shot_team_id'], '')}  {shot_label(row)}",
                 fontsize=9)
    return ax

show_state(0, 16)
plt.show()

## Every shot, per end
Each panel is the position *after* the shot; the thrown stone has a thick outline.

In [ ]:
def hammer_of(end):
    first = states[(states.end_number == end) & (states.total_shot_number == 0)]
    lead = team_by_id.get(first.iloc[0]['next_shot_team_id']) if len(first) else None
    return OTHER_TEAM.get(lead)


def show_end(end, ncols=8):
    end_states = states[(states.end_number == end) & (states.total_shot_number > 0)]
    end_states = end_states.drop_duplicates('total_shot_number', keep='last')
    n = len(end_states)
    nrows = max(1, int(np.ceil(n / ncols)))
    fig, axs = plt.subplots(nrows, ncols, figsize=(2.0 * ncols, 3.6 * nrows), squeeze=False)

    first = states[(states.end_number == end) & (states.total_shot_number == 0)]
    prev = first.iloc[0]['stones'] if len(first) else None
    for ax, (_, row) in zip(axs.flat, end_states.iterrows()):
        draw_sheet(ax)
        draw_stones(ax, row['stones'], highlight=changed_stone(prev, row['stones']))
        prev = row['stones']
        ax.set_title(f"#{row['total_shot_number']} {team_by_id.get(row['prev_shot_team_id'], '')}\n{shot_label(row)}",
                     fontsize=8)
    for ax in axs.flat[n:]:
        ax.axis('off')

    hammer = hammer_of(end)
    sc = ''
    if end < n_ends:
        sc = f"   score  team0 {score['team0'][end]} - {score['team1'][end]} team1"
    fig.suptitle(f"End {end + 1}   hammer: {hammer} ({team_names.get(hammer)}){sc}", y=1.0)
    handles = [plt.Line2D([], [], marker='o', ls='', markersize=9, markerfacecolor=TEAM_COLOR[t],
                          markeredgecolor=EDGE, label=f'{t}: {team_names[t]}') for t in TEAM_COLOR]
    fig.legend(handles=handles, loc='upper right', ncol=2, frameon=False, fontsize=8)
    fig.tight_layout()
    plt.show()

for end in sorted(states.end_number.unique()):
    show_end(end)

## Final position of each end

In [ ]:
ends = sorted(states.end_number.unique())
fig, axs = plt.subplots(1, len(ends), figsize=(2.4 * len(ends), 4.2), squeeze=False)
for ax, end in zip(axs.flat, ends):
    row = states[states.end_number == end].iloc[-1]
    draw_sheet(ax)
    draw_stones(ax, row['stones'])
    s0, s1 = (score['team0'][end], score['team1'][end]) if end < n_ends else ('-', '-')
    ax.set_title(f'End {end + 1}\nteam0 {s0} - {s1} team1', fontsize=9)
fig.tight_layout()
plt.show()

## DC3-style pixel grid (training input)

`dc4learning.sheet.to_sheet_grid`: same conversion as `PositionToPixel` in `convert_log.ipynb`
(1 px = 0.0254 m, `i` from the hog line, `j = 93 - x/0.0254`). Channel 0 = non-hammer, 1 = hammer.

In [ ]:
END, SHOT = 0, 16
row = states[(states.end_number == END) & (states.total_shot_number == SHOT)].iloc[-1]
hammer = hammer_of(END)
sheet = to_sheet_grid(row['stones'], hammer)

fig, ax = plt.subplots(figsize=(3.6, 6.4))
for ch, team in enumerate([OTHER_TEAM[hammer], hammer]):
    ii, jj = np.nonzero(sheet[ch])
    ax.scatter(jj, ii, c=TEAM_COLOR[team], edgecolors=EDGE, s=40,
               label=f"{team}{' (hammer)' if team == hammer else ''}")
ax.plot((93, 93), (0, 330), lw=0.3, c='grey')
ax.plot((0, 187), (252, 252), lw=0.3, c='grey')
ax.plot((0, 187), (324, 324), lw=0.3, c='grey')
for r in [6, 4, 2, 0.5]:
    ax.add_patch(plt.Circle((93, 252), radius=12 * r, edgecolor='grey', facecolor='none', lw=0.3))
ax.set_xlim(0, 187)
ax.set_ylim(336, 0)
ax.set_aspect('equal')
ax.legend(fontsize=8)
ax.set_title(f'End {END + 1} shot {SHOT} (pixel grid)', fontsize=9)
plt.show()

## Shot parameters

In [ ]:
shots = states[states['translational_velocity'].notna()].copy()
shots['team'] = shots['shot_team_id'].map(team_by_id)
shots['angle_deg'] = np.degrees(shots['shot_angle'])
shots['shot_idx'] = np.arange(len(shots))

fig, axs = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for team, g in shots.groupby('team'):
    axs[0].plot(g['shot_idx'], g['translational_velocity'], 'o', ms=5, c=TEAM_COLOR[team],
                mec=EDGE, mew=0.5, label=f'{team}: {team_names[team]}')
    axs[1].plot(g['shot_idx'], g['angle_deg'], 'o', ms=5, c=TEAM_COLOR[team], mec=EDGE, mew=0.5)
for e in sorted(shots.end_number.unique())[1:]:
    x0 = shots[shots.end_number == e]['shot_idx'].min() - 0.5
    for ax in axs:
        ax.axvline(x0, c='grey', lw=0.5)
axs[0].set_ylabel('velocity [m/s]')
axs[1].set_ylabel('shot angle [deg]')
axs[1].set_xlabel('shot (vertical lines = end boundaries)')
axs[0].legend(fontsize=8, frameon=False)
for ax in axs:
    ax.grid(alpha=0.3, lw=0.5)
fig.tight_layout()
plt.show()

shots[['end_number', 'total_shot_number', 'team', 'translational_velocity', 'angular_velocity', 'angle_deg',
       'actual_translational_velocity', 'actual_angular_velocity', 'actual_shot_angle']]